# ImagePlotter

Two-panel image plotter for a two-image-per-shot scan, with one **`mode`** switch:

| `mode` | Left panel | Right panel |
|---|---|---|
| `1` | image 1 of selected shot | image 2 of selected shot |
| `2` | average of all image 1 | image 1 of selected shot |
| `3` | average of all image 1 | average of all image 2 |

Each sequence writes `num_images` consecutive frames to `imgs`. For sequence index `k`:
**image 1** = frame `all_first_indices[k]` (= `num_images * k`), **image 2** = frame `+ 1`.

Both panels share the **same Gaussian smoothing and the same colorbar (vmin/vmax)** so they are
directly comparable. The zoom **crop box is optional** (set `crop_box = None` to disable). If detection
`logicals` are present in the file, the **detected atom number** is printed and shown in each title.
Figures are exported as PNG + PDF into `scan_dir/figures/`.

## 0. Setup

In [ ]:
%matplotlib qt
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.ndimage import gaussian_filter

from yb_analysis.detection import hist_init as hi
from yb_analysis.analysis.load_data import load_images, load_scan_from_path

## 1. Configure

In [ ]:
scan_dir = r"D:\OneDrive - Harvard University\Documents - Yb\Data\20260529\data_20260529_172622"

# ---- What to plot ----
mode = 1            # 1: img1 vs img2 (shot) | 2: avg(img1) vs shot img1 | 3: avg(img1) vs avg(img2)
seq_idx = 0         # which shot (used by modes 1 and 2)
n_avg = 200         # how many shots to average (used by modes 2 and 3)

# ---- Shared display knobs (applied to BOTH panels) ----
smooth_sigma = 0.80   # Gaussian filter sigma in px; set 0 to disable smoothing
vmin, vmax = None, None   # shared colorbar limits; None => auto from joint 1/99.5 percentiles
cmap = 'magma'            # 'gray', 'Purples', 'magma', ...

# ---- Crop box for the zoom-in (same coords on both panels) ----
# (y0, y1, x0, x1) in pixels — set to None to skip cropping entirely.
crop_box = (1410, 1580, 510, 680)
crop_outline_kwargs = dict(edgecolor='cyan', facecolor='none',
                           linewidth=1.2, linestyle='--')

# ---- Export knobs ----
save_dir = os.path.join(scan_dir, 'figures')
dpi = 600
save_formats = ('png', 'pdf')

## 2. Load scan context + detection logicals

In [ ]:
ctx = hi.load_scan_context(scan_dir)
print(f"num_seq={ctx['num_seq']}, num_images_per_seq={ctx['num_images']}, "
      f"total_frames={ctx['total_frames']}")

if ctx['num_images'] < 2:
    raise ValueError(
        f"This scan has num_images={ctx['num_images']} per shot; "
        'ImagePlotter expects at least 2 images per shot.')

if not (0 <= seq_idx < ctx['num_seq']):
    raise IndexError(f"seq_idx={seq_idx} out of range [0, {ctx['num_seq']})")

# Detection occupancy ('logicals'), if the file has it. Two-array files store per-image
# logicals_img1/_img2 (one row per shot); single-array files store an interleaved per-frame
# 'logicals' (one row per frame). Either may be absent (None) — then atom counts are skipped.
_sd = load_scan_from_path(scan_dir)
two_array = bool(_sd.get('two_array', False))
L1 = _sd.get('logicals_img1')
L2 = _sd.get('logicals_img2')
L = _sd.get('logicals')
has_logicals = any(a is not None for a in (L1, L2, L))
print(f"logicals available: {has_logicals} (two_array={two_array})")

## 3. Build the two panels for the selected `mode`

In [ ]:
first_idx = ctx['all_first_indices']
idx1 = first_idx[seq_idx]          # frame of image 1 of the selected shot
idx2 = idx1 + 1                    # frame of image 2 of the selected shot


def _shot_image(which):
    """Single-shot image (which=1 or 2) of the selected shot, as float64 (H, W)."""
    fidx = idx1 if which == 1 else idx2
    return load_images(ctx['data_path'], [fidx])[0].astype(np.float64)


def _avg_image(which, n):
    """Average of the first n image-1 (which=1) or image-2 (which=2) frames."""
    n = min(ctx['num_seq'], n)
    offset = 0 if which == 1 else 1
    frames = [first_idx[k] + offset for k in range(n)]
    imgs = load_images(ctx['data_path'], frames).astype(np.float64)
    return imgs.mean(axis=0), n


def _count_shot(which, k):
    """Detected atom number in image `which` of shot k, or None if unavailable."""
    if two_array:
        arr = L1 if which == 1 else L2
        if arr is None or k >= len(arr):
            return None
        return int(np.asarray(arr[k]).sum())
    if L is None:
        return None
    fidx = first_idx[k] + (0 if which == 1 else 1)
    if fidx >= len(L):
        return None
    return int(np.asarray(L[fidx]).sum())


def _count_avg(which, n):
    """Mean detected atom number per shot in image `which` over the first n shots, or None."""
    if two_array:
        arr = L1 if which == 1 else L2
        if arr is None:
            return None
        m = min(n, len(arr))
        return float(np.asarray(arr[:m]).sum(axis=1).mean())
    if L is None:
        return None
    frames = [first_idx[k] + (0 if which == 1 else 1) for k in range(min(n, ctx['num_seq']))]
    frames = [fr for fr in frames if fr < len(L)]
    if not frames:
        return None
    return float(np.asarray(L[frames]).sum(axis=1).mean())


def _count_label(kind, which, k_or_n):
    """Short ' — N atoms' suffix for a panel title (empty string if no logicals)."""
    if kind == 'shot':
        c = _count_shot(which, k_or_n)
        return '' if c is None else f'  \u2014 {c} atoms'
    c = _count_avg(which, k_or_n)
    return '' if c is None else f'  \u2014 mean {c:.1f} atoms'


# Select left/right panel content per mode. Each entry: (image, label, filename_stem).
if mode == 1:
    left = (_shot_image(1),
            f'Image 1 (shot {seq_idx}, frame {idx1}){_count_label("shot", 1, seq_idx)}',
            f'img1_shot{seq_idx}_frame{idx1}')
    right = (_shot_image(2),
             f'Image 2 (shot {seq_idx}, frame {idx2}){_count_label("shot", 2, seq_idx)}',
             f'img2_shot{seq_idx}_frame{idx2}')
elif mode == 2:
    avg1, n_used = _avg_image(1, n_avg)
    left = (avg1,
            f'Average image 1 ({n_used} shots){_count_label("avg", 1, n_avg)}',
            f'avg_img1_n{n_used}')
    right = (_shot_image(1),
             f'Image 1 (shot {seq_idx}, frame {idx1}){_count_label("shot", 1, seq_idx)}',
             f'img1_shot{seq_idx}_frame{idx1}')
elif mode == 3:
    avg1, n_used = _avg_image(1, n_avg)
    avg2, _ = _avg_image(2, n_avg)
    left = (avg1,
            f'Average image 1 ({n_used} shots){_count_label("avg", 1, n_avg)}',
            f'avg_img1_n{n_used}')
    right = (avg2,
             f'Average image 2 ({n_used} shots){_count_label("avg", 2, n_avg)}',
             f'avg_img2_n{n_used}')
else:
    raise ValueError(f'mode must be 1, 2 or 3, got {mode}')

left_img, left_label, left_stem = left
right_img, right_label, right_stem = right

# ---- Shared Gaussian smoothing ----
if smooth_sigma and smooth_sigma > 0:
    left_img = gaussian_filter(left_img, sigma=smooth_sigma)
    right_img = gaussian_filter(right_img, sigma=smooth_sigma)
    sm_tag = f'sm{str(smooth_sigma).replace(".", "p")}'   # e.g. 'sm0p8'
    sm_label = f', \u03c3={smooth_sigma}'
else:
    sm_tag = 'raw'
    sm_label = ''

# ---- Shared colorbar limits (same vmin/vmax on both panels) ----
if vmin is None or vmax is None:
    joint = np.concatenate([left_img.ravel(), right_img.ravel()])
    p_lo, p_hi = np.percentile(joint, [1, 99.5])
    v0 = p_lo if vmin is None else vmin
    v1 = p_hi if vmax is None else vmax
else:
    v0, v1 = vmin, vmax

stem = f'mode{mode}_{left_stem}__{right_stem}_{sm_tag}'
print(f'mode {mode}: left={left_label!r}  right={right_label!r}')
print(f'shared colorbar limits: [{v0:.2f}, {v1:.2f}]   smoothing: {sm_tag}')
if not has_logicals:
    print('(no detection logicals in this file — atom counts omitted)')

## 4. Plot + export

In [ ]:
os.makedirs(save_dir, exist_ok=True)


def _save_fig(fig, name):
    """Write a figure to disk in every configured format."""
    for ext in save_formats:
        path = os.path.join(save_dir, f'{name}.{ext}')
        fig.savefig(path, dpi=dpi, bbox_inches='tight')
        print(f'  wrote {path}')


def _save_single(img, title, name, draw_crop=False):
    """Render one image on its own figure (shared v0/v1) and write it to disk."""
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(img, cmap=cmap, vmin=v0, vmax=v1)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if draw_crop and crop_box is not None:
        y0, y1, x0, x1 = crop_box
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, **crop_outline_kwargs))
    fig.tight_layout()
    _save_fig(fig, name)
    return fig, ax


# ---- 1. Combined preview (crop outline drawn if crop_box is set) ----
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, img, label in ((axes[0], left_img, left_label), (axes[1], right_img, right_label)):
    im = ax.imshow(img, cmap=cmap, vmin=v0, vmax=v1)
    ax.set_title(f'{label}{sm_label}  [{v0:.1f}, {v1:.1f}]')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if crop_box is not None:
        y0, y1, x0, x1 = crop_box
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, **crop_outline_kwargs))
fig.suptitle(os.path.basename(scan_dir))
fig.tight_layout()
tag = 'with_cropbox' if crop_box is not None else 'full'
print('Saving combined preview:')
_save_fig(fig, f'{stem}_{tag}')
plt.show()

# ---- 2. Export each full image individually at high resolution ----
print('Saving full-resolution exports:')
_save_single(left_img, f'{left_label}{sm_label}', f'{left_stem}_{sm_tag}', draw_crop=True)
_save_single(right_img, f'{right_label}{sm_label}', f'{right_stem}_{sm_tag}', draw_crop=True)

# ---- 3. Crop both panels with identical coords, save and preview ----
if crop_box is not None:
    y0, y1, x0, x1 = crop_box
    left_crop = left_img[y0:y1, x0:x1]
    right_crop = right_img[y0:y1, x0:x1]
    print(f'Crop box (y0:y1, x0:x1) = ({y0}:{y1}, {x0}:{x1}) \u2192 shape {left_crop.shape}')

    print('Saving cropped exports:')
    _save_single(left_crop, f'{left_label}{sm_label} \u2014 crop [{y0}:{y1}, {x0}:{x1}]',
                 f'{left_stem}_{sm_tag}_crop_y{y0}-{y1}_x{x0}-{x1}')
    _save_single(right_crop, f'{right_label}{sm_label} \u2014 crop [{y0}:{y1}, {x0}:{x1}]',
                 f'{right_stem}_{sm_tag}_crop_y{y0}-{y1}_x{x0}-{x1}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, img, label in ((axes[0], left_crop, left_label), (axes[1], right_crop, right_label)):
        im = ax.imshow(img, cmap=cmap, vmin=v0, vmax=v1)
        ax.set_title(f'{label} \u2014 crop')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()

## 5. Export full images WITHOUT crop box

In [ ]:
print('Saving full-resolution exports (no crop box):')
_save_single(left_img, f'{left_label}{sm_label}', f'{left_stem}_{sm_tag}_nocrop', draw_crop=False)
_save_single(right_img, f'{right_label}{sm_label}', f'{right_stem}_{sm_tag}_nocrop', draw_crop=False)
plt.show()